# pipeline — Modular Inference + Consolidation (Option A: pipeline bundle)
Loads `models/182_model.pkl`, `183_model.pkl`, `184_model.pkl` according to `config/run_config.yaml`, runs only the active subset, writes each model's normalized output to its `OUT` folder with error isolation, then bundles the router into `models/final_model.pkl`.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path(os.getcwd()).resolve()
while not (ROOT / "lib").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("repo root:", ROOT)

In [ ]:
from pathlib import Path

import yaml

cfg_path = ROOT / "config" / "run_config.yaml"
with open(cfg_path) as fh:
    RUN = yaml.safe_load(fh)
print(
    "run config:", {k: RUN[k] for k in ("active_models", "input_source", "eval_mode")}
)
TH = RUN.get("thresholds", {})

## 4.2 / 4.3 Router with lazy loading + error isolation

In [ ]:
from pathlib import Path

import lib.artifacts as art
import lib.io_utils as io
from lib.pipeline_bundle import PipelineBundle

# lazy-load only active artifacts
active = {int(k): v for k, v in RUN["active_models"].items() if v}
loaded = {}
for mid, on in active.items():
    if not on:
        continue
    p = io.MODELS_DIR / f"{mid}_model.pkl"
    if p.exists():
        obj, hdr = art.load_model(p)
        loaded[mid] = obj
        print(f"loaded {mid} model (hash {hdr.get('hash')})")
    else:
        print(f"WARNING: {p} missing - run {mid}.ipynb first")
print("active models to run:", list(loaded.keys()))

## 4.4 Consolidation + 4.5 final_model.pkl

In [ ]:
bundle = PipelineBundle(RUN, loaded)
summary = bundle.run()

consolidated = {
    "title": "CASHNET consolidated dashboard",
    "active_models": list(loaded.keys()),
    "metrics": {str(m): summary["per_model"][m] for m in loaded},
    "total_routing_actions": len(summary["consolidated_actions"]),
    "routing_action_list": summary["consolidated_actions"],
    "errors": summary["errors"],
}
# surface run result in SUMMERY.txt (error isolation requirement)
summary_txt = io.repo_root() / "SUMMERY.txt"
with open(summary_txt, "w", encoding="utf-8") as fh:
    fh.write("CASHNET pipeline run\n")
    fh.write(f"active models: {list(loaded.keys())}\n")
    fh.write(f"total routing actions: {len(summary['consolidated_actions'])}\n")
    fh.write(f"errors: {summary['errors']}\n")
print("consolidation written to", summary_txt)

# Option A: serialize the bundle so final_model.pkl alone can re-run the pipeline
final_path = io.MODELS_DIR / "final_model.pkl"
art.save_model(
    bundle, final_path, provenance={"option": "A", "active": list(loaded.keys())}
)
print("saved", final_path)

## 4.6 eval_mode (optional)
When `eval_mode: true`, the same run additionally reports held-out metrics already stored in each artifact's provenance `metrics` block.

In [ ]:
if RUN.get("eval_mode"):
    print("=== evaluation report (held-out metrics from training) ===")
    for mid, _model in loaded.items():
        prov = art.load_model(io.MODELS_DIR / f"{mid}_model.pkl")[1].get(
            "provenance", {}
        )
        print(mid, prov.get("metrics", {}))
else:
    print("eval_mode is false — skipping metrics report")